In [1]:
from utils.utils import get_offers
import vertexai
import json
from vertexai.language_models import ChatModel, InputOutputTextPair

### old prompt

In [2]:
CONTEXT = """En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles pour les adolescents et jeunes adultes (15-20 ans), 
votre tâche consiste à décrire avec soin les offres mises en avant de manière convaincante et captivante, d'imaginer le contexte emotionel ou l'état d'esprit qui leur sont associées 
et de les accompagner de descriptions détaillées. 
Chaque offre doit mettre en avant les meilleurs aspects de l'expérience et fournir un contexte aux utilisateurs qui pourraient être intéressés par la réservation. 
Afin de répondre précisément à mes besoins, veuillez répondre à ces questions sous format JSON object:
- Commencez par présenter l'offre et résumez brièvement ses points forts.
Fournissez une description détaillée de l'offre, en mettant en avant ses caractéristiques uniques, ses activités ou attractions particulières qui la distinguent.
Placez l'offre dans son contexte en décrivant le type d'émotion ou d'ambiance qu'elle procure, expliquant pourquoi un utilisateur serait intéressé à la réserver pour une occasion ou une expérience spécifique.
Listez cinq exemples de titres de recommandations dans lesquelles cette offre pourrait se trouver.
Listez cinq exemples d'humeurs, de sentiments ou d'ambiances associées a cette offre.
Listez cinq moments, événements, lieux ou cette offre pourrait apparaitre.
Listez cinq autres type d'offres qui iraient bien avec cette offre en essayant de diversifier les supports.
Listez cinq moments quotidiens et evenements de la vie d'adolescents ou de jeunes adultes ou qui seraient le plus propice pour profiter de cette offre.
"""

# Entity extraction prompt

In [53]:
EXTRACT_ENTITIES = """En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles, 
votre tâche consiste à analyser avec soin les offres culturelles afin d'en extaire les entités culturelles. Ces entitées peuvent être des:
- une liste des artistes et leur metier sous format {"nom d'artiste": "métier"} présents dans l'offre, voici des exemples de metier:  (écrivain, metteur en scène,acteur , réalisateur, performer, groupe de musique, peintre, photographe, etc...), 
- une liste des disciplines artistiques majeures et champs culturels présents dans l'offre, par exemple (philosophie, peinture, littérature, cinéma, danse, sculpture, dessin, musique, mode,etc...)
- une liste des lieux ou évènements présents dans l'offre, par exemple (musées, monuments, festivals, rétrospectives, etc...)
- une liste des oeuvres ou des personnages principaux présents dans l'offre,
- une liste des spécificités des pratiques artistiques et culturelles ou les thème abordés dans l'offre, par exemple (courant artistique, genre musical, genre cinématographique, etc...).
Afin de répondre précisément à mes besoins, veuillez répondre à ces questions sous format JSON object en associant a chacune de ces typologie d'entités une liste d'un ou plusieurs résultats s'ils existent sinon associer la valeur []. N'inventez pas !
"""

## Models

In [54]:
parameters = {
    #"candidate_count": 1,
    "max_output_tokens": 2048,
    "temperature": 0,
    "top_p": 0.9,
    "top_k": 40
}

In [63]:
model_name = "chat-bison"

vertexai.init(project="passculture-data-ehp", location="us-central1")
extract_model = ChatModel.from_pretrained(model_name)

extract = extract_model.start_chat(
    context=EXTRACT_ENTITIES,
    **parameters)


chat_model = ChatModel.from_pretrained(model_name)
old = chat_model.start_chat(
    context=CONTEXT,
    **parameters)

# offers examples

In [48]:
offer_list = get_offers(number_offers=500)

Downloading: 100%|██████████|


In [49]:
offer_list[0]

{'offer_name': "L'Exorciste - Dévotion - VF",
 'offer_description': "Un jour, Angela et son amie Katherine disparaissent dans les bois avant de refaire surface 72 heures plus tard sans le moindre souvenir de ce qui leur est arrivé... Dès lors, d’étranges événements s’enchaînent et le père d'Angela doit affronter de redoutables forces maléfiques.\nTous les détails du film sur AlloCiné: https://www.allocine.fr/film/fichefilm_gen_cfilm=289029.html",
 'offer_type_domain': 'MOVIE',
 'offer_type_labels': [],
 'offer_sub_type_label': None,
 'author': None,
 'performer': None,
 'search_group_name': 'FILMS_SERIES_CINEMA',
 'is_numerical': False,
 'offer_is_duo': True,
 'offer_type_label': None,
 'subcategory_id': 'SEANCE_CINE',
 'is_underage_recommendable': True}

In [81]:
PROMPT_WORKSHOP= """
{ "titre": "atelier artistique à visée philosophique",
"description": "un temps de création artistique (art, journal, écriture) suite à un débat à visée philosophique. la réflexion philo sert de support à la créativité." ,
"category": null,
"subcategory": "ATELIER_PRATIQUE_ART"}"""

## Comparaison prompts

In [82]:

response = extract.send_message(PROMPT_WORKSHOP, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")

{"offer_name": "One Piece, le meilleur manga ?", "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 épisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une légende, tout simplement. Alors entrez Dans La Légende de One Piece", "offer_type_domain": null, "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": true, "offer_is_duo": false, "offer_type_label": null, "subcategory_id": "VOD", "is_underage_recommendable": true}
____________________
Response from Model:  {"artistes": [], "disciplines": ["philosophie"], "lieux": [], "oeuvres": [], "themes": []}


In [84]:
response = old.send_message(PROMPT_WORKSHOP, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")

____________________
Response from Model:  {
 "titre": "atelier artistique à visée philosophique",
 "description": "un temps de création artistique (art, journal, écriture) suite à un débat à visée philosophique. la réflexion philo sert de support à la créativité.",
 "category": "ATELIER_PRATIQUE",
 "subcategory": "ATELIER_PRATIQUE_ART",
 "offer_recommendation_titles": [
  "L'art de la philosophie",
  "La philosophie de l'art",
  "L'art de la pensée",
  "La pensée de l'art",
  "L'art de la création"
 ],
 "offer_recommendation_moods": [
  "Créatif",
  "Philosophique",
  "Réfléchi",
  "Curieux",
  "Ouvert d'esprit"
 ],
 "offer_recommendation_moments": [
  "Après l'école",
  "Le week-end",
  "Pendant les vacances",
  "Le soir",
  "Le matin"
 ],
 "offer_recommendation_others": [
  "Atelier de peinture",
  "Atelier de sculpture",
  "Atelier de dessin",
  "Atelier d'écriture",
  "Atelier de musique"
 ],
 "offer_recommendation_daily_moments": [
  "Après l'école",
  "Le week-end",
  "Pendant l

In [83]:
OFFER = json.dumps(offer_list[3], ensure_ascii=False )
print(OFFER)
response = old.send_message(OFFER, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")

{"offer_name": "One Piece, le meilleur manga ?", "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 épisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une légende, tout simplement. Alors entrez Dans La Légende de One Piece", "offer_type_domain": null, "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": true, "offer_is_duo": false, "offer_type_label": null, "subcategory_id": "VOD", "is_underage_recommendable": true}
____________________
Response from Model:  {
 "offer_name": "One Piece, le meilleur manga ?",
 "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 épisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une légende, tout simplement. Alors entrez Dans La Légende de One Piece",
 "offer_type_domain": "MANGA",
 "offer_type_labels": [
  "MANGA",
  "ANIME"
 ],
 "offer_sub_type_label": null,
 "autho

In [66]:
OFFER = json.dumps(offer_list[3], ensure_ascii=False )
print(OFFER)
response = extract.send_message(OFFER, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")

{"offer_name": "One Piece, le meilleur manga ?", "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 épisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une légende, tout simplement. Alors entrez Dans La Légende de One Piece", "offer_type_domain": null, "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": true, "offer_is_duo": false, "offer_type_label": null, "subcategory_id": "VOD", "is_underage_recommendable": true}
____________________
Response from Model:  {"artistes": [], "disciplines": ["dessin", "cin\u00e9ma"], "lieux": [], "oeuvres": [], "themes": []}


In [67]:
response = old.send_message(OFFER, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")

____________________
Response from Model:  {
 "offer_name": "One Piece, le meilleur manga ?",
 "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 épisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une légende, tout simplement. Alors entrez Dans La Légende de One Piece",
 "offer_type_domain": "MANGA",
 "offer_type_labels": [
  "MANGA",
  "ANIME"
 ],
 "offer_sub_type_label": null,
 "author": null,
 "performer": null,
 "search_group_name": "FILMS_SERIES_CINEMA",
 "is_numerical": true,
 "offer_is_duo": false,
 "offer_type_label": null,
 "subcategory_id": "VOD",
 "is_underage_recommendable": true,
 "offer_recommendation_titles": [
  "One Piece : le manga qui a conquis le monde",
  "One Piece : l'anime qui a révolutionné le genre",
  "One Piece : la légende de Luffy",
  "One Piece : le manga qui a changé ma vie",
  "One Piece : l'anime qui m'a fait rêver"
 ],
 "offer_recommendation_moods": [
  "Aventure",
  "Action",
  "Comédie",
  "Drame",
  "M

In [68]:
OFFER = json.dumps(offer_list[2], ensure_ascii=False )
print(OFFER)
response = extract.send_message(OFFER, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")
response = old.send_message(OFFER, **parameters)
print("____________________")
print(f"Response from Model: {response.text}")


{"offer_name": "Le Consentement - VF", "offer_description": "Paris, 1985. Vanessa a treize ans lorsqu'elle rencontre Gabriel Matzneff, écrivain quinquagénaire de renom. La jeune adolescente devient l'amante et la muse de cet homme célébré par le monde culturel et politique. Se perdant dans la relation, elle subit de plus en plus violemment l’emprise destructrice que ce prédateur exerce sur elle.\nTous les détails du film sur AlloCiné: https://www.allocine.fr/film/fichefilm_gen_cfilm=285420.html", "offer_type_domain": "MOVIE", "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": false, "offer_is_duo": true, "offer_type_label": null, "subcategory_id": "SEANCE_CINE", "is_underage_recommendable": true}
____________________
Response from Model:  {"artistes": [{"nom_artiste": "Vanessa", "m\u00e9tier": "actrice"}, {"nom_artiste": "Gabriel Matzneff", "m\u00e9tier": "\u00e9crivain"}], "disciplines":

In [80]:
print("response blocked:",response.is_blocked)
print("Blocking reasons:",response.safety_attributes )
print(f"Raw output:\n{response.raw_prediction_response}" )


response blocked: True
Blocking reasons: {}
Full output:
Prediction(predictions=[{'citationMetadata': [None], 'candidates': [{'author': 'bot', 'content': "I'm not able to help with that, as I'm only a language model. If you believe this is an error, please send us your feedback."}], 'safetyAttributes': [{'blocked': True, 'errors': [240.0]}]}], deployed_model_id='', model_version_id='', model_resource_name='', explanations=None)


### safety restriction -> violence ?

In [85]:
blocked_response = """{
  \"artistes\": [],
  \"disciplines\": [],
  \"lieux_evenements\": [],
  \"oeuvres_heros\": [],
  \"specificites\": []
}"""

## Learning examples

In [58]:
examples=[
        InputOutputTextPair(
            input_text="""\"\"\" Voici l\'offre:
 {\'offer_name\': \'Le Prince cruel : une terrible rencontre\',
  \'offer_description\': \"Enlevée au monde des mortels lorsqu\'elle n\'était qu\'une enfant, Jude vit parmi les Faes, des créatures sublimes, immortelles... et cruelles. Mais être humaine à Terrafe est un défi incessant. Et savoir manier l\'épée, maîtriser les usages, se protéger des sortilèges, tout cela ne suffit pas. \\n\\nD\'autant que Jude s\'est fait un ennemi de choix : le Prince Cardan, héritier de la couronne. Pour gagner sa place à la cour, appartenir vraiment à cette terre de magie, Jude doit le défier, quelles qu\'en soient les conséquences.\",
  \'offer_type_domain\': \'BOOK\',
  \'offer_type_labels\': [],
  \'offer_sub_type_label\': None,
  \'author\': None,
  \'performer\': None,
  \'search_group_name\': \'LIVRES\',
  \'is_numerical\': True,
  \'offer_is_duo\': False,
  \'offer_type_label\': None,
  \'subcategory_id\': \'LIVRE_NUMERIQUE\',
  \'is_underage_recommendable\': True},\"\"\"""",
            output_text="""{
  \"artistes\": ['{
    \"Holly Black\":\"écrivain\"}']
  ,
  \"disciplines\": [
    \"littérature\"
  ],
  \"lieux_evenements\": [],
  \"oeuvres_heros\": ['{\"Prince Cardan\": 'personnage'}'],
  \"specificites\": [\"fantasy\"]
}"""
        ),
        InputOutputTextPair(
            input_text="""\"\"\" Voici l\'offre:
{\'offer_name\': \'La Nonne 2 dans les Cinémas Pathé \',
  \'offer_description\': "1 place pour La Nonne 2 \nValable dans tous les cinémas Pathé et Gaumont de France CinéCarte utilisable sur le site internet, sur l’application mobile des Cinémas Pathé Gaumont, en borne ou en caisse de votre cinéma. La place est utilisable pour toutes les séances hors Séances Spéciales, hors Salles Premium et hors compléments tels que lunettes 3D, séances 3D, IMAX, 4DX, Dolby Cinema... Carte non prolongeable, non remboursable. Revente interdite. Consulter les Conditions Générales d\'Utilisation.",
  \'offer_type_domain\': \'MOVIE\',
  \'offer_type_labels\': [],
  \'offer_sub_type_label\': None,
  \'author\': None,
  \'performer\': None,
  \'search_group_name\': \'FILMS_SERIES_CINEMA\',
  \'is_numerical\': True,
  \'offer_is_duo\': False,
  \'offer_type_label\': None,
  \'subcategory_id\': \'CINE_VENTE_DISTANCE\',
  \'is_underage_recommendable\': False},\"\"\"""",
            output_text="""{
  \"artistes\": [],
  \"disciplines\": [
    \"cinéma\"
  ],
  \"lieux_evenements\": [],
  \"oeuvres_heros\": ['{\"La Nonne 2\": 'personnage'}'],
  \"specificites\": []
}"""
        ),
        InputOutputTextPair(
            input_text= """\"\"\"voici l'offre: 
{\'offer_name\': \'Haute couture\',
  \'offer_description\': "À travers l\'histoire de la mode\n\nLa mode fait partie du patrimoine culturel français et lui confère une image d\'exception depuis plusieurs siècles. Les grandes maisons de couture telles que Dior, Saint-Laurent, Chanel ou encore Jean-Paul Gaultier sont connues partout dans le monde. \n\nCette collection retrace l\'histoire et l\'actualité de ces établissements à la renommée mondiale.",
  \'offer_type_domain\': None,
  \'offer_type_labels\': [],
  \'offer_sub_type_label\': None,
  \'author\': None,
  \'performer\': None,
  \'search_group_name\': \'FILMS_SERIES_CINEMA\',
  \'is_numerical\': True,
  \'offer_is_duo\': False,
  \'offer_type_label\': None,
  \'subcategory_id\': \'VOD\',
  \'is_underage_recommendable\': True},\"\"\"""",
            output_text="""{
  \"artistes\": [],
  \"disciplines\": [
    \"cinéma\",\"mode\"
  ],
  \"lieux_evenements\": [],
  \"oeuvres_heros\": ['{\"Dior\": \"maison de couture\"}',
    '{\"Saint Laurent\": \"maison de couture\"}',
    '{\"Chanel\": \"maison de couture\"}',
    '{\"Jean Paul Gaultier\": \"maison de couture\"}'],
  \"specificites\": [\"mode\"]
}"""
        )
    ]

In [59]:
model_name = "chat-bison"

vertexai.init(project="passculture-data-ehp", location="us-central1")
fsl_extract_model = ChatModel.from_pretrained(model_name)


fsl_extract = fsl_extract_model.start_chat(
    context=EXTRACT_ENTITIES,
    examples=examples,**parameters)

In [61]:
OFFER = json.dumps(offer_list[2])
print(OFFER)
response = fsl_extract.send_message(OFFER, **parameters)
print(f"Response from Model: {response.text}")

{"offer_name": "Le Consentement - VF", "offer_description": "Paris, 1985. Vanessa a treize ans lorsqu'elle rencontre Gabriel Matzneff, écrivain quinquagénaire de renom. La jeune adolescente devient l'amante et la muse de cet homme célébré par le monde culturel et politique. Se perdant dans la relation, elle subit de plus en plus violemment l’emprise destructrice que ce prédateur exerce sur elle.\nTous les détails du film sur AlloCiné: https://www.allocine.fr/film/fichefilm_gen_cfilm=285420.html", "offer_type_domain": "MOVIE", "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": false, "offer_is_duo": true, "offer_type_label": null, "subcategory_id": "SEANCE_CINE", "is_underage_recommendable": true}
Response from Model:  {
  "artistes": [
    "{'Vanessa':'actrice'}",
    "{'Gabriel Matzneff':'écrivain'}"
  ],
  "disciplines": [
    "cin\u00e9ma"
  ],
  "lieux_evenements": [],
  "oeuvres_hero

In [62]:
OFFER = json.dumps(offer_list[2])
print(OFFER)
response = fsl_extract.send_message(OFFER, **parameters)
print(f"Response from Model: {response.text}")

{"offer_name": "One Piece, le meilleur manga ?", "offer_description": "450 millions d'exemplaires vendus dans le monde, 900 \u00e9pisodes d'anime : One Piece est plus qu'un monument de la culture manga. Une l\u00e9gende, tout simplement. Alors entrez Dans La L\u00e9gende de One Piece", "offer_type_domain": null, "offer_type_labels": [], "offer_sub_type_label": null, "author": null, "performer": null, "search_group_name": "FILMS_SERIES_CINEMA", "is_numerical": true, "offer_is_duo": false, "offer_type_label": null, "subcategory_id": "VOD", "is_underage_recommendable": true}
Response from Model:  {
  "artistes": [],
  "disciplines": [
    "cin\u00e9ma"
  ],
  "lieux_evenements": [],
  "oeuvres_heros": [],
  "specificites": []
}


## Top offers

In [86]:
entities= []
for item in offer_list:
    PROMPT = f"""voici l'offre:\n{item}"""
    response = chat.send_message(PROMPT, **parameters)
    if response.is_blocked:
        entities.append(blocked_response)
    else:
        entities.append(f"{response}")

KeyboardInterrupt: 

## Context enrichment

In [89]:
import wikipedia
wikipedia.set_lang("fr")

def wikiPageRequest(query,summary=False,sentences = 5):
    result = wikipedia.search(query, results=1,suggestion=True)
    if result[1] is None and len(result)==0:
        return None
    elif result[1] is None:
        try:
            return wikipedia.page(result[0][0]).content if not summary else wikipedia.summary(result[0][0],sentences=sentences)
        except:
            return None
    else:
        try:
            return wikipedia.page(result[1]).content if not summary else wikipedia.summary(result[1],sentences=sentences)
        except:
            return None
        
def get_context(str_entity,summary=False,sentences = 5):
    #entities = [(key,item) for key,value in eval(str_entity.text).items() for item in value if len(value)>0]
    entities = [item for key,value in eval(str_entity).items() for item in value if len(value)>0]
    entities= [ent if type(ent)==str else ent[0] for ent in entities]
    documents= []
    for ent in entities :
        documents.append(wikiPageRequest(ent,summary=summary,sentences = sentences))
        
    return documents




In [95]:
n_summary = 10
i = 1
offer = offer_list[i]
offer_entities = entities[i]
print(f"Offer:\n{offer}\n-Entities:\n{offer_entities}\n-Context:\n{get_context(offer_entities,summary=True,sentences=n_summary)}")


Offer:
{'offer_name': 'Lena Situations : découvrir son livre', 'offer_description': 'Depuis le jour de sa sortie le 24 septembre dernier, Toujours plus, paru chez Robert Laffont caracole en tête des ventes. L’ouvrage est écrit par une primo-autrice de 22 ans, Léna Situations, influenceuse nouvelle génération.', 'offer_type_domain': 'BOOK', 'offer_type_labels': [], 'offer_sub_type_label': None, 'author': None, 'performer': None, 'search_group_name': 'LIVRES', 'is_numerical': True, 'offer_is_duo': False, 'offer_type_label': None, 'subcategory_id': 'LIVRE_NUMERIQUE', 'is_underage_recommendable': True}
-Entities:
 {
  "artistes": [
    "{'Lena Situations': 'écrivain'}"
  ],
  "disciplines": [
    "littérature"
  ],
  "lieux_evenements": [],
  "oeuvres_heros": [],
  "specificites": []
}
-Context:
["Frédéric Beigbeder, né le 21 septembre 1965 à Neuilly-sur-Seine, est un écrivain, critique littéraire, scénariste, animateur de télévision et réalisateur français.\nIl est le créateur du prix de 

In [96]:
CONTEXT_PROMPT = f"""En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles pour les adolescents et jeunes adultes (15-20 ans), 
votre tâche consiste à décrire avec soin les offres mises en avant de manière convaincante et captivante, d'imaginer le contexte emotionel ou l'état d'esprit qui leur sont associées 
et de les accompagner de descriptions détaillées. En vous appuyant sur les documents suivants {get_context(offer_entities,summary=True,sentences=n_summary)}
Fournissez une description brève et détaillée de l'offre suivante en mettant en avant ses caractéristiques uniques, ses activités ou attractions particulières qui la distinguent.
"""
enrich_model = ChatModel.from_pretrained("chat-bison")
chat = enrich_model.start_chat(
    context=CONTEXT_PROMPT,**parameters)

In [98]:
response = chat.send_message(f"""{offer}""", **parameters)
print(f"Response from Model: {response.text}")

Response from Model:  Léna Situations est une youtubeuse française née le 19 novembre 1997 à Paris. Elle est l'une des influenceuses les plus populaires en France avec plus de 3,5 millions d'abonnés sur YouTube et 4,5 millions de followers sur Instagram.

Son livre, Toujours plus, est sorti le 24 septembre 2020 aux éditions Robert Laffont. Il s'agit d'un livre autobiographique dans lequel Léna Situations raconte son parcours, de son enfance à son ascension fulgurante sur les réseaux sociaux.

Le livre a été un succès commercial, se vendant à plus de 100 000 exemplaires en quelques semaines. Il a également été bien accueilli par la critique, qui a salué la sincérité et l'authenticité de Léna Situations.

Dans Toujours plus, Léna Situations aborde des sujets tels que le harcèlement scolaire, la dépression et l'anxiété. Elle parle également de son rapport à la célébrité et de la façon dont elle gère sa vie personnelle et professionnelle.

Le livre est une source d'inspiration pour tous ce

#### hallucinations Frédéric Beigbeder

In [100]:
CONTEXT2 = f"""En tant qu'expert rédacteur éditorial spécialisé dans la création de listes recommandation d'offres d'activitées et de pratiques culturelles pour les adolescents et jeunes adultes (15-20 ans), 
votre tâche consiste à décrire avec soin les offres mises en avant de manière convaincante et captivante, d'imaginer le contexte emotionel ou l'état d'esprit qui leur sont associées 
et de les accompagner de descriptions détaillées. En vous appuyant sur les documents suivants {get_context(offer_entities,summary=True,sentences=n_summary)}
Fournissez une description brève et détaillée de l'offre suivante en mettant en avant ses caractéristiques uniques, ses activités ou attractions particulières qui la distinguent.
Fournissez une description détaillée de l'offre, en mettant en avant ses caractéristiques uniques, ses activités ou attractions particulières qui la distinguent.
Placez l'offre dans son contexte en décrivant le type d'émotion ou d'ambiance qu'elle procure, expliquant pourquoi un utilisateur serait intéressé à la réserver pour une occasion ou une expérience spécifique.
Listez cinq exemples d'humeurs, de sentiments ou d'ambiances associées a cette offre.
Listez cinq moments quotidiens et evenements de la vie d'adolescents ou de jeunes adultes ou qui seraient le plus propice pour profiter de cette offre.
"""

In [101]:
enrich_model2 = ChatModel.from_pretrained("chat-bison")
chat = enrich_model2.start_chat(
    context=CONTEXT2,**parameters)

In [102]:
response = chat.send_message(f"""{offer}""", **parameters)
print(f"Response from Model: {response.text}")

Response from Model:  Toujours plus est un livre écrit par Léna Situations, une influenceuse française de 22 ans. 

Ce livre est un témoignage de la vie de Léna Situations, de son enfance à aujourd'hui. Elle y raconte son parcours, ses doutes, ses réussites et ses échecs. 

Ce livre est une source d'inspiration pour les jeunes qui se sentent perdus ou qui ont du mal à trouver leur place dans le monde. Il leur montre qu'il est possible de réussir, même si l'on ne vient pas d'un milieu favorisé. 

Ce livre est également une source de divertissement. Léna Situations a un style d'écriture léger et humoristique, ce qui rend son livre très agréable à lire. 

Si vous êtes à la recherche d'un livre inspirant et divertissant, alors Toujours plus est fait pour vous. 

Voici cinq exemples d'humeurs, de sentiments ou d'ambiances associées à cette offre : 

- Inspiration
- Motivation
- Rire
- Bonheur
- Succès

Voici cinq moments quotidiens et événements de la vie d'adolescents ou de jeunes adultes 